In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Load Excel file
champs_df = pd.read_html('sportsref_download.xls')[0]

# Flatten multi-level columns
champs_df.columns = ['_'.join(col).strip() for col in champs_df.columns]

# Extract relevant columns and rename
champs_df = champs_df[['Unnamed: 0_level_0_Year', 'Finals_Champion']].copy()
champs_df.columns = ['year', 'champion']

# Drop NaN rows
champs_df = champs_df.dropna(subset=['champion'])

# Encode years as integers
champs_df['year'] = champs_df['year'].astype(int)

champs_df

,year,champion
0,2026,New York Knicks
1,2025,Oklahoma City Thunder
2,2024,Boston Celtics
3,2023,Denver Nuggets
4,2022,Golden State Warriors
...,...,...
84,1951,Rochester Royals
85,1950,Minneapolis Lakers
86,1949,Minneapolis Lakers
87,1948,Baltimore Bullets


In [4]:
# Get most recent championship year per team
last_champ = champs_df.groupby('champion')['year'].max().reset_index()
last_champ.columns = ['team', 'last_championship']

# Calculate drought
last_champ['drought'] = 2026 - last_champ['last_championship']

# Sort for charting
last_champ = last_champ.sort_values('drought', ascending=False)

last_champ

,team,last_championship,drought
0,Baltimore Bullets,1948,78
23,Rochester Royals,1951,75
14,Minneapolis Lakers,1954,72
27,Syracuse Nationals,1955,71
20,Philadelphia Warriors,1956,70
26,St. Louis Hawks,1958,68
21,Pittsburgh Pipers,1968,58
17,Oakland Oaks,1969,57
29,Utah Stars,1971,55
9,Indiana Pacers,1973,53


In [5]:
# Define list of defunct/relocated teams to remove
defunct = [
    'Baltimore Bullets', 'Rochester Royals', 'Minneapolis Lakers',
    'Syracuse Nationals', 'Philadelphia Warriors', 'St. Louis Hawks',
    'Pittsburgh Pipers', 'Oakland Oaks', 'Utah Stars', 'Kentucky Colonels',
    'New York Nets', 'Seattle SuperSonics', 'Washington Bullets'
]

last_champ = last_champ[~last_champ['team'].isin(defunct)]

# Fix Knicks to previous championship year
last_champ.loc[last_champ['team'] == 'New York Knicks', 'last_championship'] = 1973
last_champ.loc[last_champ['team'] == 'New York Knicks', 'drought'] = 2026 - 1973

# Add renamed franchises
renamed = pd.DataFrame([
    {'team': 'Washington Wizards', 'last_championship': 1978, 'drought': 2026 - 1978},
    {'team': 'Sacramento Kings', 'last_championship': 1951, 'drought': 2026 - 1951},
    {'team': 'Atlanta Hawks', 'last_championship': 1958, 'drought': 2026 - 1958},
])

# Add never-won teams using founding year
never_won = pd.DataFrame([
    {'team': 'Phoenix Suns', 'last_championship': None, 'drought': 2026 - 1968},
    {'team': 'LA Clippers', 'last_championship': None, 'drought': 2026 - 1970},
    {'team': 'Utah Jazz', 'last_championship': None, 'drought': 2026 - 1974},
    {'team': 'Brooklyn Nets', 'last_championship': None, 'drought': 2026 - 1976},
    {'team': 'Orlando Magic', 'last_championship': None, 'drought': 2026 - 1989},
    {'team': 'Minnesota Timberwolves', 'last_championship': None, 'drought': 2026 - 1989},
    {'team': 'Charlotte Hornets', 'last_championship': None, 'drought': 2026 - 1988},
    {'team': 'Memphis Grizzlies', 'last_championship': None, 'drought': 2026 - 1995},
    {'team': 'New Orleans Pelicans', 'last_championship': None, 'drought': 2026 - 2002},
])

# Combine all and sort
last_champ = pd.concat([last_champ, renamed, never_won], ignore_index=True)
last_champ = last_champ.sort_values('drought', ascending=False).reset_index(drop=True)

# Fill never won teams as 0
last_champ['last_championship'] = last_champ['last_championship'].fillna(0)

last_champ

,team,last_championship,drought
0,Sacramento Kings,1951,75
1,Atlanta Hawks,1958,68
2,Phoenix Suns,0,58
3,LA Clippers,0,56
4,Indiana Pacers,1973,53
5,New York Knicks,1973,53
6,Utah Jazz,0,52
7,Brooklyn Nets,0,50
8,Portland Trail Blazers,1977,49
9,Washington Wizards,1978,48


In [6]:
# Highlight knicks
last_champ['color'] = last_champ['team'].apply(
    lambda x: '#179fdb' if x == 'New York Knicks' else '#C9C9C9'
)

# Create drought bar chart
drought_bar = alt.Chart(last_champ).mark_bar().encode(
    x=alt.X('drought:Q', title='Years since last championship'),
    y=alt.Y('team:N', sort='-x', title=None),
    color=alt.Color('color:N', scale=None)
).properties(width=350, height=450)

# Add source manually in properties
drought_bar = alt.layer(drought_bar).properties(
    title=alt.TitleParams(
        'Source: Basketball Reference',
        orient='bottom',
        fontStyle='italic',
        fontSize=10,
        color='#676A8680',
        fontWeight='normal',
        frame='group',
        dx=-150,
        offset=7
    )
)

# Add title as mark_text layer
title_layer = alt.Chart({'values': [{}]}).mark_text(
    text="53 Years in the Making: The Knicks End One of the NBA's Longest Championship Droughts",
    align='left',
    baseline='top',
    fontSize=14
).encode(
    x=alt.value(-150),
    y=alt.value(-25)
)

drought_bar = alt.layer(title_layer, drought_bar)

drought_bar

alt.LayerChart(...)

The New York Knicks ended a 53-year championship drought over the weekend, their first title since 1973. They are one of nine NBA franchises to have waited over 50 years for a championship.

In [7]:
# Save chart
styles.save(drought_bar, 'visualisation', 'nba_drought_bar', width=450, height=360)